# Análisis Exploratorio de Datos (EDA) — FD001

**Objetivo:** Identificar sensores constantes, patrones de degradación y distribución del RUL
en el subconjunto FD001 del conjunto de datos C-MAPSS.

**FD001:** 100 motores, 1 condición operativa (nivel del mar), 1 modo de falla (degradación HPC).

### Referencias
- Saxena et al. (2008). *Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation.* PHM08.
- Scientific Reports (2025). *Machine Learning-Based Prediction of RUL.*
- Sensors (2023). *A Hybrid Method for RUL Estimation.*

In [ ]:
# Celda 2: Imports y Configuración
# Propósito: Cargar librerías y configurar estilo publicación para visualizaciones.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Semilla para reproducibilidad
np.random.seed(42)

# Configuración de estilo publicación
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
})

sns.set_palette('colorblind')

# Paleta de colores
COLORS = {
    'primary': '#2196F3',
    'secondary': '#4CAF50',
    'accent': '#FF9800',
    'danger': '#F44336',
    'neutral': '#9E9E9E',
}

# Crear directorio para figuras
os.makedirs('figures', exist_ok=True)

print('Librerías cargadas y estilo configurado.')

In [ ]:
# Celda 3: Definición de columnas
# Propósito: Definir los 26 nombres de columna según la documentación C-MAPSS.
# Por qué: El archivo no tiene headers, hay que definirlos explícitamente evita errores de indexing.
# Referencia: datos/datos.md — estructura de columnas.

COLUMN_NAMES = [
    'unit_number',          # 1:  Número de unidad (ID del motor)
    'time_in_cycles',       # 2:  Tiempo en ciclos operativos
    'setting_1',            # 3:  Configuración operativa 1
    'setting_2',            # 4:  Configuración operativa 2
    'setting_3',            # 5:  Configuración operativa 3
    'sensor_1',             # 6:  Sensor 1 (T2 — Temperature at fan inlet)
    'sensor_2',             # 7:  Sensor 2 (T24 — Total temperature at LPC outlet)
    'sensor_3',             # 8:  Sensor 3 (T30 — Total temperature at HPC outlet)
    'sensor_4',             # 9:  Sensor 4 (T50 — Total temperature at LPT outlet)
    'sensor_5',             # 10: Sensor 5 (P2 — Pressure at fan inlet)
    'sensor_6',             # 11: Sensor 6 (P15 — Total pressure in bypass-duct)
    'sensor_7',             # 12: Sensor 7 (P30 — Total pressure at HPC outlet)
    'sensor_8',             # 13: Sensor 8 (Nf — Physical fan speed)
    'sensor_9',             # 14: Sensor 9 (Nc — Physical core speed)
    'sensor_10',            # 15: Sensor 10 (epr — Engine pressure ratio)
    'sensor_11',            # 16: Sensor 11 (Ps30 — Static pressure at HPC outlet)
    'sensor_12',            # 17: Sensor 12 (phi — Ratio of fuel flow to Ps30)
    'sensor_13',            # 18: Sensor 13 (NRf — Corrected fan speed)
    'sensor_14',            # 19: Sensor 14 (NRc — Corrected core speed)
    'sensor_15',            # 20: Sensor 15 (BPR — Bypass ratio)
    'sensor_16',            # 21: Sensor 16 (farB — Burner fuel-air ratio)
    'sensor_17',            # 22: Sensor 17 (htBleed — Bleed enthalpy)
    'sensor_18',            # 23: Sensor 18 (Tfan — Fan speed)
    'sensor_19',            # 24: Sensor 19 (Tbleed — Bleed temperature)
    'sensor_20',            # 25: Sensor 20 (W31 — HPT bleed coolant bleed)
    'sensor_21',            # 26: Sensor 21 (W32 — LPT bleed coolant bleed)
]

# Listas auxiliares para indexar
SETTING_COLS = [c for c in COLUMN_NAMES if c.startswith('setting_')]
SENSOR_COLS = [c for c in COLUMN_NAMES if c.startswith('sensor_')]

print(f'Total columnas: {len(COLUMN_NAMES)}')
print(f'Configuraciones operativas: {len(SETTING_COLS)}')
print(f'Sensores: {len(SENSOR_COLS)}')

In [ ]:
# Celda 4: Carga de datos
# Propósito: Cargar train_FD001.txt y verificar dimensiones.

DATA_PATH = '../datos/train_FD001.txt'

# Cargar con sep=r'\s+' para manejar múltiples espacios y trailing whitespace
df = pd.read_csv(DATA_PATH, sep=r'\s+', header=None, names=COLUMN_NAMES)

print(f'Dimensiones: {df.shape}')
print(f'Motores: {df["unit_number"].nunique()}')
print(f'Ciclos totales: {len(df):,}')
print(f'\nPrimeras filas:')
df.head(10)

In [ ]:
# Celda 5: Últimas filas
print(f'\nÚltimas filas:')
df.tail(10)

In [ ]:
# Celda 6: Información general
# Propósito: Explorar tipos de datos, valores nulos y estadísticas descriptivas.
# Por qué: Detectar problemas de calidad de datos (nulos, tipos incorrectos) antes del análisis.

print('=== TIPOS DE DATOS ===')
print(df.dtypes)
print(f'\n=== VALORES NULOS ===')
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else 'No hay valores nulos.')
print(f'\n=== ESTADÍSTICAS DESCRIPTIVAS (sensores) ===')
df[SENSOR_COLS].describe().round(4).T

In [ ]:
# Celda 7: Cálculo y distribución del RUL
# Propósito: Calcular RUL = max(ciclos) - ciclos_actuales por motor.
# Por qué: El RUL es la variable objetivo; entender su distribución es clave para modelado.

# RUL = vida útil restante = ciclos máximos del motor - ciclos actuales
max_cycles = df.groupby('unit_number')['time_in_cycles'].transform('max')
df['RUL'] = max_cycles - df['time_in_cycles']

# Estadísticas del RUL por motor
rul_stats = df.groupby('unit_number').agg(
    ciclos_totales=('time_in_cycles', 'max'),
    rul_inicial=('RUL', 'max'),
    rul_final=('RUL', 'min')
).reset_index()

print('=== ESTADÍSTICAS DEL RUL POR MOTOR ===')
print(f'RUL promedio inicial: {rul_stats["rul_inicial"].mean():.1f} ciclos')
print(f'RUL mínimo inicial:   {rul_stats["rul_inicial"].min()} ciclos')
print(f'RUL máximo inicial:   {rul_stats["rul_inicial"].max()} ciclos')
print(f'Ciclos totales promedio: {rul_stats["ciclos_totales"].mean():.1f}')
print(f'\nDistribución de RUL global:')
df['RUL'].describe().round(2)

In [ ]:
# Celda 8: Histograma y box plot del RUL
# Propósito: Visualizar la distribución del RUL para identificar sesgo y outliers.
# Por qué: Un RUL sesgado afecta el entrenamiento de modelos; los outliers indican motores atípicos.
# Referencia: Sensors (2023) — análisis de distribución de RUL.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['RUL'], bins=30, edgecolor='black', alpha=0.7, color=COLORS['primary'])
axes[0].axvline(df['RUL'].mean(), color=COLORS['danger'], linestyle='--', linewidth=1.5,
                label=f'Media: {df["RUL"].mean():.1f}')
axes[0].axvline(df['RUL'].median(), color=COLORS['secondary'], linestyle='--', linewidth=1.5,
                label=f'Mediana: {df["RUL"].median():.1f}')
axes[0].set_xlabel('RUL (ciclos)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del RUL')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
bp = axes[1].boxplot(df['RUL'].dropna(), vert=True, patch_artist=True,
                     boxprops=dict(facecolor=COLORS['primary'], alpha=0.7))
axes[1].set_ylabel('RUL (ciclos)')
axes[1].set_title('Box Plot del RUL')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/rul_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/rul_distribution.pdf', bbox_inches='tight')
plt.show()

# Interpretación
print('=== INTERPRETACIÓN ===')
skewness = df['RUL'].skew()
print(f'Asimetría (skewness): {skewness:.3f}')
print(f'Desviación estándar: {df["RUL"].std():.1f} ciclos')
if abs(skewness) > 1:
    print('→ Distribución fuertemente sesgada. Considerar transformación (log, sqrt) para modelado.')
elif abs(skewness) > 0.5:
    print('→ Distribución moderadamente sesgada.')
else:
    print('→ Distribución aproximadamente simétrica.')

In [ ]:
# Celda 9: Análisis de varianza de sensores
# Propósito: Calcular la varianza de cada sensor para identificar cuáles son constantes y cuáles variables.
# Por qué: Sensores con varianza muy baja (≈0) no aportan información y deben eliminarse.
#          Sensores con alta varianza son candidatos a ser "sensores clave" para análisis.
# Referencia: MDFA paper (2025) — análisis de contribución de sensores.

# Calcular varianza de cada sensor
variances = df[SENSOR_COLS].var()

# Crear DataFrame con resultados
variance_analysis = pd.DataFrame({
    'sensor': variances.index,
    'varianza': variances.values,
    'std': df[SENSOR_COLS].std().values,
    'rango': df[SENSOR_COLS].max().values - df[SENSOR_COLS].min().values
}).sort_values('varianza', ascending=False)

print('=== ANÁLISIS DE VARIANZA DE SENSORES ===')
print('(ordenados de mayor a menor varianza)\n')
print(variance_analysis.to_string(index=False))

# Clasificar sensores
CONSTANT_THRESHOLD = 0.001
constant_sensors = variance_analysis[variance_analysis['varianza'] < CONSTANT_THRESHOLD]['sensor'].tolist()
variable_sensors = variance_analysis[variance_analysis['varianza'] >= CONSTANT_THRESHOLD]['sensor'].tolist()

print(f'\n=== CLASIFICACIÓN ===')
print(f'Sensores constantes (varianza < {CONSTANT_THRESHOLD}): {len(constant_sensors)}')
print(f'  → {constant_sensors}')
print(f'\nSensores variables (varianza >= {CONSTANT_THRESHOLD}): {len(variable_sensors)}')
print(f'  → {variable_sensors}')

In [ ]:
# Celda 10: Visualización de varianza (MEJORADA)
# Propósito: Visualizar la varianza de cada sensor para identificar patrones.
# Por qué: La visualización facilita la identificación de sensores constantes vs variables.
#          Usamos escala logarítmica porque las varianzas cubren varios órdenes de magnitud.

fig, ax = plt.subplots(figsize=(14, 9))

# Preparar datos ordenados por varianza
plot_data = variance_analysis.copy()
plot_data['log_varianza'] = np.log10(plot_data['varianza'].clip(lower=1e-10))
plot_data['categoria'] = plot_data['sensor'].apply(
    lambda x: 'Constante' if x in constant_sensors else 'Variable'
)

# Paleta de colores mejorada
palette = {'Constante': '#EF4444', 'Variable': '#3B82F6'}

# Crear barras con colores por categoría
colors_bar = [palette[cat] for cat in plot_data['categoria']]
bars = ax.barh(range(len(plot_data)), plot_data['varianza'], 
               color=colors_bar, edgecolor='white', linewidth=0.8, height=0.7)

# Configurar eje Y con nombres de sensores
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data['sensor'], fontsize=10)

# Escala logarítmica para el eje X
ax.set_xscale('log')

# Línea de umbral
ax.axvline(x=CONSTANT_THRESHOLD, color='#DC2626', linestyle='--', 
           linewidth=2, alpha=0.8, label=f'Umbral: {CONSTANT_THRESHOLD}')

# Etiquetas de valor con formato mejorado
for i, (bar, value, cat) in enumerate(zip(bars, plot_data['varianza'], plot_data['categoria'])):
    # Posición de la etiqueta
    label_x = bar.get_width() * 1.5 if value > 0 else 1e-8
    
    # Formato del valor según magnitud
    if value < 1e-6:
        label = f'{value:.2e}'
    elif value < 1:
        label = f'{value:.4f}'
    else:
        label = f'{value:.1f}'
    
    ax.text(label_x, i, label, va='center', fontsize=9, fontweight='bold',
            color=palette[cat])

# Leyenda y etiquetas
ax.set_xlabel('Varianza (escala logarítmica)', fontsize=12, fontweight='bold')
ax.set_title('Varianza de Sensores — Detección de Sensores Constantes', 
             fontsize=14, fontweight='bold', pad=20)

# Leyenda personalizada
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=palette['Constante'], edgecolor='white', label='Constante (var < 0.001)'),
    Patch(facecolor=palette['Variable'], edgecolor='white', label='Variable (var >= 0.001)'),
    plt.Line2D([0], [0], color='#DC2626', linestyle='--', linewidth=2, label=f'Umbral: {CONSTANT_THRESHOLD}')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10, framealpha=0.9)

# Estilo de la cuadrícula
ax.grid(True, alpha=0.3, axis='x', linestyle='-', linewidth=0.5)
ax.set_axisbelow(True)

# Ajustar límites
ax.set_xlim(left=1e-10)

# Agregar anotación con resumen
n_const = len(constant_sensors)
n_var = len(variable_sensors)
ax.text(0.02, 0.98, f'Constantes: {n_const} | Variables: {n_var}',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('figures/sensor_variance.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/sensor_variance.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Celda 11: Tasa de contribución de sensores
# Propósito: Calcular la contribución porcentual de cada sensor a la varianza total.
# Por qué: Esta métrica nos dice qué sensores son más "informativos" para el modelo.
#          Los sensores con mayor contribución son candidatos a ser "sensores clave".
# Referencia: MDFA paper (2025) — análisis de contribución de sensores.

# Calcular contribución porcentual
total_var = variances.sum()
contribution = (variances / total_var * 100).sort_values(ascending=False)

# Crear DataFrame con resultados
contribution_df = pd.DataFrame({
    'sensor': contribution.index,
    'contribucion_pct': contribution.values,
    'varianza': variances[contribution.index].values
})

print('=== CONTRIBUCIÓN PORCENTUAL A LA VARIANZA TOTAL ===')
print(f'Varianza total: {total_var:.4f}\n')
print(contribution_df.to_string(index=False))

# Identificar sensores con contribución significativa
CONTRIBUTION_THRESHOLD = 1.0  # 1%
significant_sensors = contribution_df[contribution_df['contribucion_pct'] >= CONTRIBUTION_THRESHOLD]['sensor'].tolist()
low_contrib_sensors = contribution_df[contribution_df['contribucion_pct'] < CONTRIBUTION_THRESHOLD]['sensor'].tolist()

print(f'\n=== CLASIFICACIÓN POR CONTRIBUCIÓN ===')
print(f'Sensores con contribución >= {CONTRIBUTION_THRESHOLD}%: {len(significant_sensors)}')
print(f'  → {significant_sensors}')
print(f'\nSensores con contribución < {CONTRIBUTION_THRESHOLD}%: {len(low_contrib_sensors)}')
print(f'  → {low_contrib_sensors}')

In [ ]:
# Celda 12: Visualización de contribución
# Propósito: Visualizar la contribución porcentual de cada sensor.
# Por qué: Facilita la identificación de sensores dominantes y minoritarios.

fig, ax = plt.subplots(figsize=(12, 8))

# Colores: rojo para baja contribución, azul para significativa
colors_contrib = [COLORS['danger'] if s in low_contrib_sensors else COLORS['primary'] 
                  for s in contribution_df['sensor']]

bars = ax.barh(contribution_df['sensor'], contribution_df['contribucion_pct'], 
               color=colors_contrib, edgecolor='black', linewidth=0.5)

# Línea de umbral
ax.axvline(x=CONTRIBUTION_THRESHOLD, color=COLORS['danger'], linestyle='--', 
           linewidth=1.5, label=f'Umbral: {CONTRIBUTION_THRESHOLD}%')

# Etiquetas de valor
for bar, value in zip(bars, contribution_df['contribucion_pct']):
    label_x = bar.get_width() + max(contribution_df['contribucion_pct']) * 0.01
    ax.text(label_x, bar.get_y() + bar.get_height()/2,
            f'{value:.1f}%', va='center', fontsize=9)

ax.set_xlabel('Contribución a la Varianza Total (%)')
ax.set_title('Tasa de Contribución de Sensores')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('figures/sensor_contribution.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/sensor_contribution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Celda 13: Selección de sensores clave
# Propósito: Seleccionar los sensores más informativos para visualización de degradación.
# Por qué: No podemos asumir qué sensores son "clave" sin un cálculo explícito.
#          Este paso DEMUESTRA el criterio de selección basado en los datos.

print('=== CRITERIO DE SELECCIÓN DE SENSORES CLAVE ===')
print(f'\n1. Excluir sensores constantes (varianza < {CONSTANT_THRESHOLD}):')
print(f'   → Excluidos: {constant_sensors}')
print(f'\n2. Seleccionar sensores con mayor contribución a la varianza:')
print(f'   → Umbral: contribución >= {CONTRIBUTION_THRESHOLD}%')
print(f'   → Seleccionados: {significant_sensors}')

# Validación: los sensores seleccionados deben estar en la lista de variables
key_sensors = [s for s in significant_sensors if s in variable_sensors]

print(f'\n=== SENSORES CLAVE SELECCIONADOS ===')
print(f'Cantidad: {len(key_sensors)}')
print(f'Sensores: {key_sensors}')

# Mostrar estadísticas de los sensores seleccionados
print(f'\nEstadísticas de sensores clave:')
key_stats = pd.DataFrame({
    'sensor': key_sensors,
    'varianza': [variances[s] for s in key_sensors],
    'contribucion_pct': [contribution[s] for s in key_sensors],
    'media': [df[s].mean() for s in key_sensors],
    'std': [df[s].std() for s in key_sensors]
})
print(key_stats.to_string(index=False))

print(f'\n=== JUSTIFICACIÓN ===')
print('Estos sensores fueron seleccionados porque:')
print('1. No son constantes (varianza > 0.001)')
print('2. Contribuyen significativamente a la varianza total (>= 1%)')
print('3. Son los más informativos para detectar patrones de degradación')

In [ ]:
# Celda 14: Trayectorias de degradación
# Propósito: Visualizar patrones de degradación en motores representativos usando sensores clave.
# Por qué: Los sensores clave (seleccionados en Celda 13) son los más informativos.
#          Mostramos su comportamiento a lo largo de la vida útil del motor.
# Referencia: Scientific Reports (2025) — análisis de patrones de degradación.

# Seleccionar motores representativos: vida larga, corta y media
engine_lifetimes = df.groupby('unit_number')['time_in_cycles'].max().sort_values()
n_engines = len(engine_lifetimes)

# Motores representativos: 5 más cortos, 5 medianos, 5 más largos
short_engines = engine_lifetimes.head(5).index.tolist()
long_engines = engine_lifetimes.tail(5).index.tolist()
mid_engines = engine_lifetimes.iloc[n_engines//2 - 2 : n_engines//2 + 3].index.tolist()

representative_engines = short_engines[:1] + mid_engines[:2] + long_engines[:2]
print(f'Motores representativos (vida en ciclos):')
for eid in representative_engines:
    life = engine_lifetimes[eid]
    label = 'corta' if eid in short_engines else ('media' if eid in mid_engines else 'larga')
    print(f'  Motor {eid}: {life} ciclos ({label})')

# Usar sensores clave seleccionados en Celda 13
print(f'\nSensores a graficar (seleccionados en Celda 13): {key_sensors}')

# Gráfico de trayectorias de degradación
n_sensors = len(key_sensors)
fig, axes = plt.subplots(n_sensors, 1, figsize=(14, 3 * n_sensors), sharex=True)

colors_engines = plt.cm.Set2(np.linspace(0, 1, len(representative_engines)))

for i, sensor in enumerate(key_sensors):
    for j, engine_id in enumerate(representative_engines):
        engine_data = df[df['unit_number'] == engine_id]
        axes[i].plot(engine_data['time_in_cycles'],
                     engine_data[sensor],
                     color=colors_engines[j],
                     alpha=0.7,
                     linewidth=1.5,
                     label=f'Motor {engine_id}')

    axes[i].set_ylabel(sensor, fontsize=10)
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(labelsize=9)

    if i == 0:
        axes[i].legend(loc='upper right', fontsize=8, ncol=2)

axes[-1].set_xlabel('Ciclos operativos')
plt.suptitle('Trayectorias de Degradación — Sensores Clave', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/degradation_trajectories.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/degradation_trajectories.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Celda 15: Análisis de correlación
# Propósito: Identificar pares de sensores altamente correlacionados (|r| > 0.8).
# Por qué: Sensores altamente correlacionados son redundantes; eliminarlos reduce dimensionalidad.
# Referencia: Scientific Reports (2025) — análisis de correlación multivariada.

# Matriz de correlación
corr_matrix = df[SENSOR_COLS].corr()

# Encontrar pares con correlación > 0.8
high_corr_pairs = []
for i in range(len(SENSOR_COLS)):
    for j in range(i + 1, len(SENSOR_COLS)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append({
                'Sensor 1': SENSOR_COLS[i],
                'Sensor 2': SENSOR_COLS[j],
                'Correlación': round(corr_matrix.iloc[i, j], 4)
            })

print('=== PARES CON CORRELACIÓN > 0.8 ===')
if high_corr_pairs:
    corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlación', key=abs, ascending=False)
    print(corr_df.to_string(index=False))
else:
    print('No se encontraron pares con correlación > 0.8.')

# Heatmap de correlación
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)

sns.heatmap(corr_matrix, mask=mask, cmap=cmap, center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8}, ax=ax, annot_kws={'size': 7})

ax.set_title('Mapa de Correlación de Sensores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/correlation_heatmap.pdf', bbox_inches='tight')
plt.show()

## Resumen Ejecutivo

### Hallazgos Principales

1. **Datos:** 100 motores, 20,631 observaciones, 21 sensores + 3 configuraciones operativas.
2. **RUL:** Distribución sesgada a la derecha (skewness > 0.5), media ~107 ciclos, rango [0, 361].
3. **Sensores constantes:** 7 sensores con varianza ≈ 0 (sensores 1, 5, 6, 10, 16, 18, 19).
4. **Sensores clave:** Seleccionados por contribución a la varianza (>= 1%): sensor_9, sensor_14, sensor_4, sensor_3.
5. **Degradación:** Sensores clave muestran tendencias claras de degradación a lo largo de la vida útil.
6. **Correlación:** Pares altamente correlacionados detectados (|r| > 0.8) — redundancia potencial.

### Criterio de Selección de Sensores Clave

Los sensores "clave" NO se asumieron; se seleccionaron mediante:
1. **Exclusión de constantes:** Varianza < 0.001 → eliminados.
2. **Ranking por contribución:** Contribución a varianza total >= 1% → seleccionados.
3. **Validación:** Los seleccionados deben estar en la lista de sensores variables.

### Recomendaciones para Preprocesamiento

- **Eliminar sensores constantes** (varianza < 0.001) — no aportan información.
- **Eliminar sensores con contribución < 1%** — baja información relativa.
- **Evaluar multicolinealidad** — pares con |r| > 0.8, considerar eliminar uno de cada par.
- **Transformar RUL** — considerar transformación log o raíz cuadrada para sesgo.
- **Normalizar sensores** — MinMax o StandardScaler antes de modelado.

### Próximos Pasos

- notebooks/02_preprocessing_FD001.ipynb — limpieza y transformación de datos.
- notebooks/03_modeling_FD001.ipynb — entrenamiento de modelos RUL.
- Evaluar FD002, FD003, FD004 para comparar patrones entre subconjuntos.